

# Session 1 · Agentic AI Fundamentals

**Course: Agentic AI and Multi-Agent Systems** · Fundacion AI Granada
Monday 13 July 2026 · 10:00-12:00

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/montevive/agentic-ai-course/blob/main/session-1-agent-fundamentals.ipynb)

| Time | Block |
|---|---|
| 10:00 | From LLM to agent: the paradigm shift |
| 10:30 | Anatomy of an agent: perception, reasoning, memory, action |
| 11:00 | Live implementation: a working agent from scratch (Pydantic AI, smolagents, OpenAI Agents SDK) |
| 11:40 | Technical discussion and exercise: applications in research and teaching |

### What you'll learn

By the end of this session you will be able to:

1. Explain why a bare LLM call fails in real settings (stateless, no actions, frozen knowledge).
2. Describe the ReAct pattern and the native tool-use contract, and implement the agent loop by hand.
3. Identify the components of an agent (reasoning, perception, memory, tools + execution environment) and their cost implications.
4. Build a typed, tool-using agent with structured output and self-correction in Pydantic AI.
5. Compare Pydantic AI, smolagents and the OpenAI Agents SDK, and recognize when a fixed workflow beats an agent.

Running the whole notebook costs well under 0.50 EUR in API calls with the default (cheap) models.

The **Solutions** section for the exercises is at the end of the notebook.

## Setup

Run this cell first. It detects your environment (local or Colab), loads the API keys from `.env` (or Colab Secrets) and picks the working model based on the available provider.

> **Note on `await`:** Pydantic AI is async-first, and Jupyter runs cells inside an event loop, so you can write `await agent.run(...)` directly in a cell. In a plain `.py` script you would wrap it in `asyncio.run(...)` or call the synchronous `agent.run_sync(...)` instead.

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip install -q -r https://raw.githubusercontent.com/montevive/agentic-ai-course/main/requirements.txt
    if not Path("data").exists():
        !git clone --depth 1 https://github.com/montevive/agentic-ai-course.git /content/agentic-ai-course
        %cd /content/agentic-ai-course

from dotenv import load_dotenv
load_dotenv()

if IN_COLAB:
    from google.colab import userdata
    for key in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_API_KEY"):
        try:
            value = userdata.get(key)
            if value:
                os.environ[key] = value
        except Exception:
            pass

PROVIDERS = {
    "anthropic": bool(os.getenv("ANTHROPIC_API_KEY")),
    "openai": bool(os.getenv("OPENAI_API_KEY")),
    "google": bool(os.getenv("GOOGLE_API_KEY")),
}

if PROVIDERS["anthropic"]:
    MODEL, MODEL_CHEAP = "anthropic:claude-sonnet-4-6", "anthropic:claude-haiku-4-5"
elif PROVIDERS["openai"]:
    MODEL, MODEL_CHEAP = "openai:gpt-5", "openai:gpt-5-mini"
elif PROVIDERS["google"]:
    MODEL, MODEL_CHEAP = "google-gla:gemini-pro-latest", "google-gla:gemini-flash-latest"
else:
    raise RuntimeError("No API keys. Check the 00-environment-check notebook.")

print("Available providers:", [k for k, v in PROVIDERS.items() if v])
print(f"Working model: {MODEL}")
print(f"Low-cost model: {MODEL_CHEAP}")

---

## 10:00 · Block 1: from LLM to agent

An LLM is a function: text in, text out. It doesn't look anything up, doesn't run anything, doesn't remember anything. Let's start by seeing where that breaks.

### 1.1 · The static call (and its limits)

We ask a *bare* model for two things: a large multiplication and a question about a private corpus of papers it obviously never saw in training.

In [ ]:
from pydantic_ai import Agent

plain_llm = Agent(MODEL_CHEAP)

r = await plain_llm.run("What is 987654 * 123457? Reply with the number only.")
print("Model:", r.output)
print("Real: ", 987654 * 123457)

In [ ]:
r = await plain_llm.run(
    "According to the paper 2403-multiagent-coordination in our internal corpus, "
    "how much does hierarchical coordination improve over a single agent?"
)
print(r.output)

The model either gets the arithmetic wrong or just makes things up (or admits it doesn't know). Neither failure is a bug: an LLM is a *stateless next-token predictor* over a finite context window. It doesn't execute algorithms (numbers are just tokens, so long multiplications drift) and it can't look anything up (its knowledge is frozen at training time). **A model on its own is not enough in real settings**: it needs tools to act and a loop to decide when to use them.

### 1.2 · From Chain-of-Thought to ReAct

**Chain-of-Thought** ([Wei et al., 2022](https://arxiv.org/abs/2201.11903)) was the first fix: ask the model to reason step by step before answering. Accuracy improves a lot, but the model is still talking to itself: it can't *verify* anything or *fetch* anything.

**ReAct** ([Yao et al., 2022](https://arxiv.org/abs/2210.03629)) closes the loop: it interleaves **reasoning** with **actions** (tool calls) and **observations** (their results). The original paper did it purely with prompting; the trace looks like this:

```
Thought: I need the numbers from the coordination paper.
Action: search_corpus("multi-agent coordination quality")
Observation: [2403-multiagent-coordination] ...hierarchical +18% over single agent...
Thought: I have the figure. Now I can answer.
Answer: hierarchical coordination, +18% over a single agent.
```

![From a single LLM call to the agent loop](https://raw.githubusercontent.com/montevive/agentic-ai-course/main/assets/s1-from-llm-to-agent.svg)

Modern APIs bake this pattern in as **native tool use**: instead of parsing `Action: ...` out of free text, the model emits structured tool-call blocks and the API tells you when it wants to act (section 1.4). Every framework we'll see this week (Pydantic AI, LangGraph, CrewAI...) is a variation on this loop. Before using any of them, we'll **write it by hand** so nothing feels like magic.

> **When NOT to build an agent.** If you already know the steps (fetch, then summarize, then format), a fixed *workflow* is cheaper, faster and more predictable than an agent ([Anthropic, "Building effective agents"](https://www.anthropic.com/engineering/building-effective-agents)). Agents earn their cost when the path has to be decided at runtime, step by step. Keep this in mind tomorrow, when it gets tempting to throw a team of agents at everything.

### 1.3 · The tools

Two example tools we'll work with all session:

- `search_corpus`: term search over the mini-corpus of abstracts in `data/`
- `calculator`: exact arithmetic

In [ ]:
import re

CORPUS = {p.stem: p.read_text(encoding="utf-8") for p in sorted(Path("data").glob("2*.md"))}
print(f"{len(CORPUS)} papers in the corpus:", ", ".join(list(CORPUS)[:4]), "...")

def search_corpus(query: str) -> str:
    terms = [t for t in re.findall(r"\w+", query.lower()) if len(t) > 3]
    scored = sorted(
        ((sum(text.lower().count(t) for t in terms), name) for name, text in CORPUS.items()),
        reverse=True,
    )
    top = [(s, n) for s, n in scored[:3] if s > 0]
    if not top:
        return "No results for that query."
    return "\n\n".join(f"[{name}]\n{CORPUS[name][:700]}" for _, name in top)

# eval() behind a character allowlist is fine for a demo, but never eval
# untrusted input in real systems - Session 3 covers sandboxing.
def calculator(expression: str) -> str:
    if not set(expression) <= set("0123456789+-*/(). "):
        return "Error: only arithmetic expressions are allowed."
    return str(eval(expression))

print()
print(calculator("987654 * 123457"))

### 1.4 · The agent loop written by hand

No frameworks: the raw Anthropic API. The contract is simple:

1. We declare the tools with a **JSON Schema** of their parameters.
2. If the model wants to use one, it responds with `stop_reason == "tool_use"`.
3. **We** run the tool and return the result as a `tool_result`.
4. Repeat until the model responds with final text.

![The tool-use contract between your code, the model API and your tools](https://raw.githubusercontent.com/montevive/agentic-ai-course/main/assets/s1-tool-use-sequence.svg)

The same contract exists in every provider, only the field names change: `tool_use` blocks in Anthropic, `tool_calls` in OpenAI, `function_call` in Gemini. This block uses the Anthropic API.

In [ ]:
if not PROVIDERS["anthropic"]:
    print("⚪ No ANTHROPIC_API_KEY: block demonstrated in class. The pattern is identical on OpenAI/Gemini.")
else:
    import anthropic

    client = anthropic.Anthropic()

    TOOLS = [
        {
            "name": "search_corpus",
            "description": "Search papers in the local corpus of abstracts on agentic AI. Use it for any question about the corpus literature.",
            "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
        },
        {
            "name": "calculator",
            "description": "Evaluate an arithmetic expression exactly.",
            "input_schema": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]},
        },
    ]
    TOOL_FUNCTIONS = {"search_corpus": search_corpus, "calculator": calculator}

    def run_agent(question: str, max_steps: int = 8) -> str:
        messages = [{"role": "user", "content": question}]
        for step in range(1, max_steps + 1):
            response = client.messages.create(
                model="claude-haiku-4-5", max_tokens=1024, tools=TOOLS, messages=messages,
            )
            messages.append({"role": "assistant", "content": response.content})
            if response.stop_reason != "tool_use":
                return next(b.text for b in response.content if b.type == "text")
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  [step {step}] {block.name}({block.input})")
                    output = TOOL_FUNCTIONS[block.name](**block.input)
                    tool_results.append(
                        {"type": "tool_result", "tool_use_id": block.id, "content": output}
                    )
            messages.append({"role": "user", "content": tool_results})
        return "Step limit reached without a final answer."

    print(run_agent(
        "According to the corpus, which multi-agent coordination pattern wins on quality and by what "
        "percentage does it improve over a single agent? And by the way: what is 987654 * 123457?"
    ))

Notice what just happened:

- The model **decided on its own** which tool to use, when and with what arguments.
- We only ran the tools and returned results: **the harness (our code) controls execution**, the model controls the strategy.
- The whole agent "state" is the `messages` list. Hold on to that idea for block 2.

About 30 lines of loop. Everything else we'll see this week is layers of convenience, robustness and scale on top of this.

A useful mental model for researchers: the loop is a *policy*. The state is the `messages` list, the action space is "call a tool" or "emit the final answer", and observations are tool results. Agents are sequential decision-making with an LLM as the policy, which is why evaluating them (Session 3) borrows so much from experimental methodology.

---

## 10:30 · Block 2: anatomy of an agent

![Anatomy of an agent](https://raw.githubusercontent.com/montevive/agentic-ai-course/main/assets/s1-agent-anatomy.svg)

| Component | What it is | In our hand-written loop |
|---|---|---|
| **Reasoning** | The LLM as the decision engine | `client.messages.create(...)` |
| **Perception** | What enters the context | `messages` (question + tool results) |
| **Short-term memory** | The conversation history | the `messages` list |
| **Long-term memory** | Knowledge persisted across sessions | *we don't have any (Session 2: RAG and Mem0)* |
| **Tools** | The ability to act | `TOOLS` + `TOOL_FUNCTIONS` |
| **Execution environment** | Where the actions run | this notebook (in production: sandbox) |

Two components deserve a closer look before we run anything:

- **Execution environment**: where tool code actually runs. Here it's this notebook's Python process, with no isolation: a buggy or hijacked tool can do anything your user can. In production, tools run in sandboxes with permissions, timeouts and resource limits (Session 3).
- **The context window is a budget**: everything the agent perceives (history, tool results, instructions) competes for the same finite window, and you pay per token for it on every call. Managing it deliberately is called *context engineering* (Session 2).

> **Long-term memory, in one paragraph.** Anything the agent should keep *across* conversations lives outside the model, and there are two families: **retrieval** over documents (RAG: index a corpus, fetch relevant chunks into the context on demand) and **memory layers** that extract and consolidate facts from past interactions ("Ana works on time series"). Today's agent has neither; tomorrow we build both (ChromaDB and Mem0, Session 2).

### 2.1 · Short-term memory IS the message list

The API is *stateless*: the model remembers nothing between calls. The "memory" is that **you resend the full history** every time.

In [ ]:
chat = Agent(MODEL_CHEAP, instructions="You are a concise assistant.")

r1 = await chat.run("My name is Ana and my research is about time series applied to energy.")
print("1>", r1.output)

r2 = await chat.run("What do I research?", message_history=r1.new_messages())
print("2 (with history)>", r2.output)

r3 = await chat.run("What do I research?")
print("3 (no history) >", r3.output)

In [ ]:
print(f"Messages accumulated in history: {len(r2.all_messages())}")
print(f"Token usage for turn 2: {r2.usage}")

Direct (and very expensive if ignored) consequences:

- The history **grows** with each turn and each tool result -> more tokens -> more cost and latency.
- The context window is finite: for long tasks you must **prune, compact or offload** memory.
- That is exactly the *context engineering* topic of Session 2.

### 2.2 · The other two ingredients

- **Tools**: we've already seen them. The quality of the tool *description* matters as much as its code: it's the only thing the model reads to decide whether to use it.
- **Execution environment**: in the notebook we run tools without a safety net. In production: sandbox, permissions, time limits and auditing (Session 3).

---

## 11:00 · Block 3: live implementation with frameworks

The hand-written loop teaches the mechanism, but as soon as you want **structured output, retries, validation and multi-provider**, a framework pays off. Let's start with a lightweight one.

### 3.1 · Pydantic AI: a typed agent with structured output

Three key ideas:

1. `output_type`: the agent returns a **validated Pydantic object**, not free text.
2. `@agent.tool_plain`: any Python function is a tool (the signature and docstring generate the schema on their own).
3. `ModelRetry`: if a tool fails, the error goes back to the model so it **self-corrects**.

> **Why structured output changes everything.** Free text is for humans; validated objects are for *pipelines*. With `output_type`, generation is guided by the JSON Schema of your Pydantic model, and every response is validated before it reaches your code. When validation fails, the error is sent back to the model so it can correct itself. That validate-retry loop is the same error-handling idea you'll reuse for tools (`ModelRetry`) and for output guardrails in Session 3.

![The validate-retry loop behind structured output](https://raw.githubusercontent.com/montevive/agentic-ai-course/main/assets/s1-structured-output-retry.svg)

In [ ]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent, ModelRetry

class Citation(BaseModel):
    paper: str = Field(description="Paper identifier, e.g. 2403-multiagent-coordination")
    fact: str = Field(description="Specific fact from the paper that supports the answer")

class ResearchAnswer(BaseModel):
    answer: str
    citations: list[Citation]
    confidence: float = Field(ge=0, le=1)

research_agent = Agent(
    MODEL,
    output_type=ResearchAnswer,
    retries=2,
    instructions=(
        "You are a research assistant. Answer ONLY with information from the local corpus, "
        "using your tools. If the corpus doesn't cover the question, say so with low confidence."
    ),
)

@research_agent.tool_plain
def search_in_corpus(query: str) -> str:
    """Search relevant papers in the local corpus of abstracts on agentic AI."""
    return search_corpus(query)

@research_agent.tool_plain
def read_paper(paper_id: str) -> str:
    """Return the full text of a corpus paper given its identifier."""
    if paper_id not in CORPUS:
        raise ModelRetry(
            f"The paper {paper_id!r} does not exist. Use search_in_corpus to find valid identifiers."
        )
    return CORPUS[paper_id]

In [ ]:
result = await research_agent.run(
    "Which context engineering strategies reduce agent cost and how much do they save?"
)

answer = result.output
print(answer.answer)
print()
for citation in answer.citations:
    print(f"  · [{citation.paper}] {citation.fact}")
print(f"\nconfidence: {answer.confidence}")
print(f"usage: {result.usage}")

In [ ]:
print(type(answer))
answer.model_dump()

The output is a **validated object**: if the model returns something that doesn't meet the schema (a missing field, confidence outside [0,1]...), Pydantic AI retries automatically. This is what turns an agent into something you can **actually slot into a pipeline**.

### 3.2 · smolagents (Hugging Face): the agent that writes code

A different paradigm: instead of emitting JSON tool calls, the `CodeAgent` **writes and runs Python code** as its action. For tasks with composite logic (search, filter, compute, aggregate) it usually needs fewer steps.

This is the **CodeAct** idea ([Wang et al., 2024](https://arxiv.org/abs/2402.01030)): code is a more expressive action space than one-tool-at-a-time JSON calls. The model can compose tools, loop, branch and keep intermediate variables in a single step, which means fewer round-trips. The price is that you are now executing model-written code, which is exactly why sandboxing matters (Session 3).

*(Code execution runs in a local sandbox that restricts imports; for production smolagents supports isolated remote executors.)*

In [ ]:
from smolagents import CodeAgent, OpenAIServerModel, tool

if PROVIDERS["anthropic"]:
    smol_model = OpenAIServerModel(
        model_id="claude-haiku-4-5",
        api_base="https://api.anthropic.com/v1/",
        api_key=os.environ["ANTHROPIC_API_KEY"],
    )
elif PROVIDERS["openai"]:
    smol_model = OpenAIServerModel(model_id="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])
else:
    smol_model = OpenAIServerModel(
        model_id="gemini-flash-latest",
        api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=os.environ["GOOGLE_API_KEY"],
    )

@tool
def search_papers(query: str) -> str:
    """Search papers in the local corpus of abstracts on agentic AI.

    Args:
        query: search terms, in English or Spanish.
    """
    return search_corpus(query)

smol_agent = CodeAgent(tools=[search_papers], model=smol_model, max_steps=6)

smol_agent.run(
    "Search the corpus for the integration-time reduction reported by the MCP paper "
    "(from 3.2 days to 4 hours) and compute that reduction as a percentage."
)

Look at the trace: the agent generates **code blocks** (`search_papers(...)`, computations with intermediate variables) instead of isolated tool calls. Same ReAct loop, a different representation of the action.

### 3.3 · OpenAI Agents SDK: the same agent, a third framework

To compare flavors. OpenAI's Agents SDK uses `Agent` + `Runner` and `@function_tool`. Native to OpenAI (with other providers it works via LiteLLM).

LangChain exposes the same pattern too (`llm.bind_tools(...)`); we'll use it in Session 2 through LangGraph, so by then you'll have seen all the main flavors.

In [ ]:
if not PROVIDERS["openai"]:
    print("⚪ This block uses the OpenAI API natively; without OPENAI_API_KEY it is skipped.")
    print("   With other providers: https://openai.github.io/openai-agents-python/models/litellm/")
else:
    from agents import Agent as OAIAgent, Runner, function_tool

    @function_tool
    def search_papers_openai(query: str) -> str:
        """Search papers in the local corpus of abstracts on agentic AI."""
        return search_corpus(query)

    oai_agent = OAIAgent(
        name="Research assistant",
        model="gpt-5-mini",
        instructions="Answer using the search tool over the local corpus, citing the papers.",
        tools=[search_papers_openai],
    )

    oai_result = await Runner.run(
        oai_agent, "What does the corpus say about agent memory layers versus vectorstores?"
    )
    print(oai_result.final_output)

### 3.4 · The three frameworks, in one table

| | Pydantic AI | smolagents | OpenAI Agents SDK |
|---|---|---|---|
| Action | JSON tool calls | **Python code** | JSON tool calls |
| Structured output | ✅ native (Pydantic) | via prompt | ✅ (Pydantic) |
| Multi-provider | ✅ native | ✅ (OpenAI-compat / LiteLLM) | via LiteLLM |
| Self-correction | `ModelRetry` | retries with the error | retries |
| Best for | typed pipelines | compute/data tasks | OpenAI ecosystem |

Rule of thumb:

- **Typed pipelines** whose output other code consumes -> Pydantic AI.
- **Compute- or data-heavy tasks** (filter, aggregate, calculate) -> smolagents.
- **Deep in the OpenAI ecosystem** (Responses API, hosted tools) -> OpenAI Agents SDK.

> **Local and private models.** All three speak the OpenAI-compatible protocol, so the same agents run against a local [Ollama](https://ollama.com) server (`api_base="http://localhost:11434/v1"`) or any vLLM endpoint. Relevant when your data can't leave the machine, or for zero-marginal-cost teaching labs.

### 3.5 · Note: visual prototyping with n8n

Before writing code, the same flow (trigger -> LLM -> tools -> output) can be prototyped in [n8n](https://n8n.io) by dragging nodes (*AI Agent* + *Tools*). We'll see it in a live demo outside the notebook: useful to validate an idea with non-technical people before investing in implementation.

---

## 11:40 · Block 4: discussion and exercise

**Discussion (10'):** identify ONE process in your research or teaching that an agent could automate. Think in terms of the anatomy: which tools does it need? what memory? what's the risk if it gets it wrong?

Examples from the enrollment survey: literature review, reproducible experimentation, teaching-material generation, scientific monitoring.

We'll close with a **collective synthesis**: everyone shares their candidate process, and the best ones come back on Tuesday as cases for the multi-agent workshop.

### 🧪 Exercise: your first agent with a tool of your own

Build a Pydantic AI agent with **one tool from your domain** (it can return simulated data, what matters is the design):

1. Define the role in `instructions`.
2. Implement the tool with a clear docstring (the model only sees that).
3. Optional: define a structured `output_type`.

*Example solution at the end of the notebook.*

In [ ]:
my_agent = Agent(
    MODEL_CHEAP,
    instructions="TODO: describe your agent's role (who it is, what it does, what it must NOT do)",
)

@my_agent.tool_plain
def my_tool(parameter: str) -> str:
    """TODO: describe what this tool does; the model decides to use it by reading this."""
    return "TODO: implement or simulate the result"

# Uncomment when ready:
# result = await my_agent.run("TODO: a realistic request from your day-to-day")
# print(result.output)

---

## Takeaways

1. An agent = **LLM + loop + tools + memory**. The loop fits in 30 lines.
2. Short-term memory is the message list; it grows and costs money.
3. Frameworks add types, validation, retries and multi-provider support, not magic.
4. A tool's description is part of the program: write it for the model.

**Next session (Tuesday 14):** from an agent to a team: coordination, CrewAI vs LangGraph, context engineering, memory and RAG.

## Further reading

- [Wei et al., 2022 - Chain-of-Thought Prompting](https://arxiv.org/abs/2201.11903): reasoning before answering.
- [Yao et al., 2022 - ReAct](https://arxiv.org/abs/2210.03629): the reason-act-observe loop this whole course builds on.
- [Wang et al., 2024 - Executable Code Actions (CodeAct)](https://arxiv.org/abs/2402.01030): code as the action space, the idea behind smolagents.
- [Anthropic - Building effective agents](https://www.anthropic.com/engineering/building-effective-agents): workflows vs agents, and the patterns that work in production.
- [Anthropic - Tool use documentation](https://docs.claude.com/en/docs/agents-and-tools/tool-use/overview): the raw contract from section 1.4.
- [Pydantic AI docs](https://ai.pydantic.dev/) · [smolagents docs](https://huggingface.co/docs/smolagents) · [OpenAI Agents SDK docs](https://openai.github.io/openai-agents-python/): the three frameworks you used today.

---

## Solutions

### Exercise: agent with a custom tool (example: publications assistant)

In [ ]:
from pydantic import BaseModel

SIMULATED_PUBLICATIONS = {
    "0000-0002-1234-5678": [
        {"title": "Deep learning for electricity demand forecasting", "year": 2024, "citations": 31},
        {"title": "Transformers for meteorological time series", "year": 2025, "citations": 12},
        {"title": "Benchmark of time-series foundation models", "year": 2026, "citations": 3},
    ],
}

class ProfileSummary(BaseModel):
    research_lines: list[str]
    highlighted_publication: str
    suggested_next_paper: str

publications_assistant = Agent(
    MODEL_CHEAP,
    output_type=ProfileSummary,
    instructions=(
        "You are a researcher-profile assistant. Look up the publications with your tool "
        "and synthesize research lines. Do not invent publications."
    ),
)

@publications_assistant.tool_plain
def publications_by_orcid(orcid: str) -> str:
    """Return the publications registered for an ORCID (title, year, citations)."""
    pubs = SIMULATED_PUBLICATIONS.get(orcid)
    if not pubs:
        return f"No publications for ORCID {orcid}."
    return "\n".join(f"- {p['title']} ({p['year']}, {p['citations']} citations)" for p in pubs)

sol = await publications_assistant.run(
    "Summarize the profile for ORCID 0000-0002-1234-5678 and suggest a next paper."
)
print(sol.output.model_dump_json(indent=2))

---



**Montevive AI** · [montevive.ai](https://montevive.ai) · chema@montevive.ai